# 00F — Deployment diagnostics

**Run All with no settings for read-only discovery.** Only boto3 and Python's
standard library are required. Each AWS or Docker command is visible in its
cell. Failures print the original exception and do not stop unrelated checks.
The last cell prints a compact summary for a screenshot.

Run this in Studio, then in the proposed build environment (for example Cloud9).
Each run describes that machine and its current AWS identity. Listing Cloud9
environments from Studio does not test Docker inside Cloud9.

The first bucket and repository returned are used only for read checks.
No data contents, credentials or build environment-variable values are printed.
Resource names and error messages can still identify your environment; keep
outputs there and clear them before sharing this notebook.

**Later sections have separate switches for pipeline API tests, S3 writes,
repository creation and Docker pull/build/push.** They are off by default.
Successful reads do not establish write permissions, and an API denial does
not establish that a resource is absent. Discovery reads only one page.


## 1. Current environment and optional read locations

- Use the current credentials and region; print any region fallback.
- Leave the read locations blank to use the first resource returned.
- Initialize results and unique probe names; rerun with a fresh kernel for a new probe.


In [ ]:
import base64
import json
import re
import subprocess
import sys
import tempfile
import uuid
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from pprint import pprint
from urllib.parse import urlsplit

import boto3
from botocore.config import Config

bucket_name = ""        # Optional existing bucket for reads.
s3_prefix = ""          # Optional prefix inside that bucket.
repository_name = ""   # Optional existing ECR repository in this region.

session = boto3.Session()
region = session.region_name or "us-east-1"
config = Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0})
sts = session.client("sts", region_name=region, config=config)
iam = session.client("iam", region_name=region, config=config)
s3 = session.client("s3", region_name=region, config=config)
ecr = session.client("ecr", region_name=region, config=config)
sm = session.client("sagemaker", region_name=region, config=config)
cloud9 = session.client("cloud9", region_name=region, config=config)
codebuild = session.client("codebuild", region_name=region, config=config)
cfn = session.client("cloudformation", region_name=region, config=config)
ssm = session.client("ssm", region_name=region, config=config)
results = {}
probe_id = uuid.uuid4().hex
probe_name = "deployment-check-" + probe_id[:10]
caller = {}
role_name = role_arn = object_key = existing_pipeline = build_project = None
pipeline_arn = execution_arn = written_object = created_repository = None
push_uri = built_image = pushed_digest = None
print("Python:", sys.version.split()[0], "| kernel:", sys.executable)
print("Working directory:", Path.cwd())
print("Region:", region, "(fallback)" if not session.region_name else "")
print("Probe name:", probe_name)


## 2. AWS caller — GetCallerIdentity

- Print the actual identity used by the following AWS calls.
- Derive a role candidate if these are assumed-role credentials.
- Treat the candidate's IAM path as unverified until GetRole succeeds.


In [ ]:
try:
    caller = sts.get_caller_identity()
    pprint({key: caller[key] for key in ("Account", "Arn", "UserId")})
    if ":assumed-role/" in caller["Arn"]:
        role_name = caller["Arn"].split(":assumed-role/", 1)[1].split("/")[0]
        role_arn = f"arn:{caller['Arn'].split(':')[1]}:iam::{caller['Account']}:role/{role_name}"
        print("Current role candidate (IAM path unverified):", role_arn)
    results["STS identity"] = "confirmed"
except Exception as error:
    print(error)
    results["STS identity"] = str(error)


## 3. Installed Python packages

- Read installed versions without installing anything.
- Missing SageMaker/CDK packages do not prevent these boto3 checks.
- These are notebook-kernel packages, not packages inside a container.


In [ ]:
for package in ("boto3", "sagemaker", "aws-cdk-lib", "pandas", "autogluon.tabular", "torch"):
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "not installed in this kernel")


## 4. Docker CLI

- Run docker --version on this machine.
- Print its output or command error.
- A working CLI alone does not establish a working Docker daemon.


In [ ]:
try:
    response = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=15, check=True)
    print(response.stdout)
    results["Docker CLI"] = "confirmed"
except Exception as error:
    print(getattr(error, "stderr", "") or error)
    results["Docker CLI"] = getattr(error, "stderr", "") or str(error)


## 5. Docker daemon

- Ask the current Docker context for server information.
- Print the daemon version and operating system if reachable.
- Do not start a daemon or change Studio settings.


In [ ]:
try:
    response = subprocess.run(
        ["docker", "info", "--format", "{{.ServerVersion}} | {{.OSType}} | {{.Architecture}}"],
        capture_output=True, text=True, timeout=15, check=True,
    )
    print(response.stdout)
    results["Docker daemon"] = "confirmed"
except Exception as error:
    print(getattr(error, "stderr", "") or error)
    results["Docker daemon"] = getattr(error, "stderr", "") or str(error)


## 6. Visible Cloud9 environments — ListEnvironments

- List up to ten environments visible to this identity in this region.
- An empty page or denial does not prove there is no usable build environment.
- Run this notebook inside the intended environment to test its Docker and AWS access.


In [ ]:
try:
    response = cloud9.list_environments(maxResults=10)
    pprint(response.get("environmentIds", []))
    print("More pages:", bool(response.get("nextToken")))
    results["Cloud9 list"] = "confirmed"
except Exception as error:
    print(error)
    results["Cloud9 list"] = str(error)


## 7. S3 buckets — ListBuckets

- List the first five buckets independently of pipeline permissions.
- Select the first only if no bucket was supplied in cell 1.
- Preserve a supplied bucket even if listing fails.


In [ ]:
try:
    response = s3.list_buckets(MaxBuckets=5)
    pprint([item["Name"] for item in response.get("Buckets", [])])
    if not bucket_name and response.get("Buckets"):
        bucket_name = response["Buckets"][0]["Name"]
    results["S3 ListBuckets"] = "confirmed"
except Exception as error:
    print(error)
    results["S3 ListBuckets"] = str(error)
print("Read location:", f"s3://{bucket_name}/{s3_prefix}" if bucket_name else "none")


## 8. S3 files — ListObjectsV2

- List up to ten objects under the selected bucket/prefix.
- Select the first nonempty file for a read check.
- Print names and sizes, not file contents.


In [ ]:
results["S3 ListObjectsV2"] = "not exercised: no bucket"
if bucket_name:
    try:
        response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix, MaxKeys=10)
        pprint([{"Key": item["Key"], "Size": item["Size"]} for item in response.get("Contents", [])])
        object_key = next((item["Key"] for item in response.get("Contents", [])
                           if item["Size"] and not item["Key"].endswith("/")), None)
        results["S3 ListObjectsV2"] = "confirmed"
    except Exception as error:
        print(error)
        results["S3 ListObjectsV2"] = str(error)
print("File for read:", object_key or "none discovered in this page")


## 9. S3 object read — GetObject

- Request at most the first 1 KiB of the discovered file.
- Close the response body and print only the byte count and version.
- A successful read proves access to this object, not every bucket or prefix.


In [ ]:
results["S3 GetObject"] = "not exercised: no nonempty object"
if bucket_name and object_key:
    try:
        response = s3.get_object(Bucket=bucket_name, Key=object_key, Range="bytes=0-1023")
        with response["Body"] as body:
            bytes_read = len(body.read())
        print("Bytes read:", bytes_read, "| VersionId:", response.get("VersionId"))
        results["S3 GetObject"] = "confirmed"
    except Exception as error:
        print(error)
        results["S3 GetObject"] = str(error)


## 10. ECR repositories — DescribeRepositories

- List the first ten repositories in the current account and region.
- Use the first for read checks unless a repository was supplied.
- Keep repository discovery separate from permission to push images.


In [ ]:
try:
    response = ecr.describe_repositories(maxResults=10)
    pprint([item["repositoryUri"] for item in response.get("repositories", [])])
    if not repository_name and response.get("repositories"):
        repository_name = response["repositories"][0]["repositoryName"]
    results["ECR repositories"] = "confirmed"
except Exception as error:
    print(error)
    results["ECR repositories"] = str(error)
print("Repository for reads:", repository_name or "none")


## 11. ECR images — DescribeImages

- Read up to five image records from the selected repository.
- Print image tags and digests.
- Image metadata access is separate from Docker pull or push access.


In [ ]:
results["ECR image metadata"] = "not exercised: no repository"
if repository_name:
    try:
        response = ecr.describe_images(repositoryName=repository_name, maxResults=5)
        pprint([{"digest": item["imageDigest"], "tags": item.get("imageTags", [])}
                for item in response.get("imageDetails", [])])
        results["ECR image metadata"] = "confirmed"
    except Exception as error:
        print(error)
        results["ECR image metadata"] = str(error)


## 12. ECR authentication — GetAuthorizationToken

- Request an ECR authentication token without logging Docker in.
- Print its expiry and endpoint only; discard the token.
- Token issuance does not establish access to any particular image.


In [ ]:
try:
    response = ecr.get_authorization_token()
    pprint([{"endpoint": item["proxyEndpoint"], "expires": item["expiresAt"]}
            for item in response["authorizationData"]])
    del response
    results["ECR token"] = "confirmed"
except Exception as error:
    print(error)
    results["ECR token"] = str(error)


## 13. Exact current role — GetRole

- Resolve the role inferred from STS, including any IAM path.
- Print its trust policy and permissions-boundary ARN if available.
- If IAM is denied or unreachable, other services are still checked.


In [ ]:
results["IAM GetRole"] = "not exercised: caller is not an assumed role"
if role_name:
    try:
        response = iam.get_role(RoleName=role_name)["Role"]
        role_arn = response["Arn"]
        pprint({key: response.get(key) for key in ("Arn", "AssumeRolePolicyDocument", "PermissionsBoundary")})
        results["IAM GetRole"] = "confirmed"
    except Exception as error:
        print(error)
        results["IAM GetRole"] = str(error)


## 14. Existing SageMaker role candidates — ListRoles

- Look at the first 100 roles and print ones mentioning SageMaker in their trust policy.
- These are candidates for the platform team to confirm, not automatically approved roles.
- Listing roles cannot establish PassRole, job permissions or trust-condition compatibility.


In [ ]:
try:
    response = iam.list_roles(MaxItems=100)
    pprint([item["Arn"] for item in response["Roles"]
            if "sagemaker.amazonaws.com" in json.dumps(item["AssumeRolePolicyDocument"])])
    print("More roles not inspected:", response.get("IsTruncated", False))
    results["IAM ListRoles"] = "confirmed"
except Exception as error:
    print(error)
    results["IAM ListRoles"] = str(error)


## 15. Existing pipelines — ListPipelines

- List up to ten pipelines in this region.
- Choose the first only for the next read check.
- Never update or execute a discovered existing pipeline.


In [ ]:
try:
    response = sm.list_pipelines(MaxResults=10)
    pprint([item["PipelineName"] for item in response.get("PipelineSummaries", [])])
    existing_pipeline = next((item["PipelineName"] for item in response.get("PipelineSummaries", [])), None)
    results["SageMaker ListPipelines"] = "confirmed"
except Exception as error:
    print(error)
    results["SageMaker ListPipelines"] = str(error)


## 16. Existing pipeline roles — DescribePipeline

- Read the selected pipeline's orchestration role.
- Print job roles and image references declared in its steps, including conditional branches.
- Use these references for discussion; this notebook does not reuse or alter them.


In [ ]:
results["SageMaker DescribePipeline"] = "not exercised: no pipeline discovered"
if existing_pipeline:
    try:
        response = sm.describe_pipeline(PipelineName=existing_pipeline)
        print("Pipeline:", response["PipelineArn"], "| pipeline role:", response["RoleArn"])
        steps = list(json.loads(response["PipelineDefinition"]).get("Steps", []))
        while steps:
            step = steps.pop(0)
            args = step.get("Arguments", {})
            print(step["Name"], step["Type"], "| job role:", args.get("RoleArn", ""),
                  "| image:", args.get("AppSpecification", {}).get("ImageUri",
                              args.get("AlgorithmSpecification", {}).get("TrainingImage", "")))
            steps.extend(args.get("IfSteps", []) + args.get("ElseSteps", []))
        results["SageMaker DescribePipeline"] = "confirmed"
    except Exception as error:
        print(error)
        results["SageMaker DescribePipeline"] = str(error)


## 17. Existing remote builders — ListProjects

- List the first page of CodeBuild projects and display up to ten.
- An approved Cloud9 or GitHub build route may make CodeBuild unnecessary.
- Do not start a discovered project; it could perform a full deployment.


In [ ]:
try:
    response = codebuild.list_projects()
    pprint(response.get("projects", [])[:10])
    build_project = next(iter(response.get("projects", [])), None)
    results["CodeBuild ListProjects"] = "confirmed"
except Exception as error:
    print(error)
    results["CodeBuild ListProjects"] = str(error)


## 18. Builder configuration — BatchGetProjects

- Read the first discovered project's service role, worker image and source type.
- Show privileged-mode and VPC settings without printing environment variables or secrets.
- Visibility is not permission to start builds or approval to use this project.


In [ ]:
results["CodeBuild BatchGetProjects"] = "not exercised: no project"
if build_project:
    try:
        response = codebuild.batch_get_projects(names=[build_project])
        for project in response["projects"]:
            pprint({"name": project["name"], "role": project["serviceRole"],
                    "worker_image": project["environment"]["image"],
                    "privileged": project["environment"].get("privilegedMode"),
                    "source_type": project["source"]["type"], "vpc": project.get("vpcConfig")})
        results["CodeBuild BatchGetProjects"] = "confirmed"
    except Exception as error:
        print(error)
        results["CodeBuild BatchGetProjects"] = str(error)


## 19. Standard CDK bootstrap — DescribeStacks

- Read the standard CDKToolkit stack if visible.
- A different bootstrap name or deployment process may be used by the platform.
- This is not a deployment permission test and is not needed to run pipeline APIs.


In [ ]:
try:
    response = cfn.describe_stacks(StackName="CDKToolkit")
    pprint([{"name": item["StackName"], "status": item["StackStatus"]} for item in response["Stacks"]])
    results["CloudFormation bootstrap read"] = "confirmed"
except Exception as error:
    print(error)
    results["CloudFormation bootstrap read"] = str(error)


## 20. Standard CDK bootstrap version — GetParameter

- Read the default bootstrap version parameter independently of CloudFormation.
- Print the version or original error.
- A successful read does not grant CDK deployment permissions.


In [ ]:
try:
    response = ssm.get_parameter(Name="/cdk-bootstrap/hnb659fds/version")
    print(response["Parameter"]["Name"], response["Parameter"]["Value"])
    results["SSM bootstrap read"] = "confirmed"
except Exception as error:
    print(error)
    results["SSM bootstrap read"] = str(error)


## 21. Local source and package-index configuration

- Find the nearest checkout above this notebook and print relevant file paths.
- Show configured pip index hostnames only, stripping usernames, passwords and URL paths.
- File/configuration presence does not prove GitHub workflow access or package-download access.


In [ ]:
checkout = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / ".git").exists() or (path / "mlops-models").is_dir()), None)
print("Checkout:", checkout or "not found above this notebook")
if checkout:
    for pattern in (".github/workflows/*.yml", ".github/workflows/*.yaml", "infra.ts",
                    "bin/*.ts", "package.json", "mlops-models/containers/*Dockerfile"):
        for path in sorted(checkout.glob(pattern)):
            print(path)
try:
    response = subprocess.run([sys.executable, "-m", "pip", "config", "list"],
                              capture_output=True, text=True, timeout=15, check=True)
    pip_config = dict(line.split("=", 1) for line in response.stdout.splitlines() if "=" in line)
    for key, value in pip_config.items():
        if "index-url" in key:
            print(key, [urlsplit(url.strip("'\"")).hostname for url in value.split()])
    if not any("index-url" in key for key in pip_config):
        print("No explicit pip index URL found; no download attempted.")
except Exception as error:
    print("Could not inspect pip configuration:", type(error).__name__)
    # Do not print pip's raw configuration output; it can contain credentials.


# Optional: prove pipeline create, update and start permissions

Set the switch below to true for an approved development probe. The pipeline
contains one condition (`1 == 1`) and **launches no jobs**. No image, job role,
Docker, S3 input or CDK is needed. A pipeline role is still required by the API.
Leave its override blank to try the current role; enter the platform-provided
pipeline role if appropriate. Listing/GetRole failures do not block that attempt.

This proves pipeline control-plane calls and whether this role can execute
this definition. It does **not** prove Training, Processing, Batch Transform,
Registry, job-role passing, container networking or runtime data access.


## 22. CreatePipeline — independent of container availability

- Enable this test explicitly; use the current role or a supplied pipeline-role ARN.
- Print the exact unique name and role being passed.
- Create only this probe, whose complete definition is visible below.


In [ ]:
TEST_PIPELINE = False
pipeline_role_override = ""  # Optional. Empty tries the current role candidate.
pipeline_role = pipeline_role_override or role_arn
definition = {
    "Version": "2020-12-01",
    "Steps": [{
        "Name": "CheckControlPlane", "Type": "Condition",
        "Arguments": {
            "Conditions": [{"Type": "Equals", "LeftValue": 1, "RightValue": 1}],
            "IfSteps": [], "ElseSteps": [],
        },
    }],
}
print("Pipeline:", probe_name, "| role:", pipeline_role)
results["CreatePipeline"] = "not exercised: TEST_PIPELINE is false"
if TEST_PIPELINE:
    try:
        response = sm.create_pipeline(
            PipelineName=probe_name, RoleArn=pipeline_role,
            PipelineDefinition=json.dumps(definition), ClientRequestToken=probe_id + "-create",
        )
        pipeline_arn = response["PipelineArn"]
        print(pipeline_arn)
        results["CreatePipeline"] = "confirmed"
    except Exception as error:
        print(error)
        results["CreatePipeline"] = str(error)


## 23. UpdatePipeline

- Update only the probe successfully created in this kernel.
- Change its description through an explicit API call.
- An update denial does not prevent the next start attempt.


In [ ]:
results["UpdatePipeline"] = "not exercised: probe not created"
if TEST_PIPELINE and pipeline_arn:
    try:
        response = sm.update_pipeline(PipelineName=probe_name, PipelineDescription="Updated diagnostics probe")
        print(response["PipelineArn"])
        results["UpdatePipeline"] = "confirmed"
    except Exception as error:
        print(error)
        results["UpdatePipeline"] = str(error)


## 24. StartPipelineExecution

- Start the new probe even if UpdatePipeline was denied.
- Reuse the token so retrying this cell does not start another execution.
- Print the ARN; acceptance of start is separate from successful execution.


In [ ]:
results["StartPipelineExecution"] = "not exercised: probe not created"
if TEST_PIPELINE and pipeline_arn:
    try:
        response = sm.start_pipeline_execution(PipelineName=probe_name, ClientRequestToken=probe_id + "-start")
        execution_arn = response["PipelineExecutionArn"]
        print(execution_arn)
        results["StartPipelineExecution"] = "confirmed"
    except Exception as error:
        print(error)
        results["StartPipelineExecution"] = str(error)


## 25. DescribePipelineExecution

- Read status and any failure reason from the new execution.
- Rerun this cell if the asynchronous execution is still Executing.
- A Succeeded result verifies this condition-only pipeline, not a managed model job.


In [ ]:
results["Probe execution"] = "not exercised: not started"
if execution_arn:
    try:
        response = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)
        print(response["PipelineExecutionStatus"], response.get("FailureReason", ""))
        results["Probe execution"] = response["PipelineExecutionStatus"]
    except Exception as error:
        print(error)
        results["Probe execution"] = str(error)


## 26. Optional S3 write — PutObject

- Enable only if the displayed bucket/prefix is approved for this tiny write.
- Create one unique text object; never overwrite a discovered file.
- Retain its version ID for exact cleanup, including versioned buckets.


In [ ]:
TEST_S3_WRITE = False
write_key = s3_prefix.rstrip("/") + "/" if s3_prefix else ""
write_key += "diagnostics/" + probe_id + "/probe.txt"
print("Write target:", f"s3://{bucket_name}/{write_key}")
results["S3 PutObject"] = "not exercised: TEST_S3_WRITE is false"
if TEST_S3_WRITE:
    try:
        response = s3.put_object(Bucket=bucket_name, Key=write_key, Body=b"deployment diagnostics\n")
        written_object = {"Bucket": bucket_name, "Key": write_key}
        if "VersionId" in response:
            written_object["VersionId"] = response["VersionId"]
        print(written_object)
        results["S3 PutObject"] = "confirmed"
    except Exception as error:
        print(error)
        results["S3 PutObject"] = str(error)


## 27. Read back the new S3 object

- Read only the exact object/version created above.
- Compare its bytes with the known probe text.
- A failed read does not prevent the later delete attempt.


In [ ]:
results["S3 probe readback"] = "not exercised: no probe object"
if written_object:
    try:
        response = s3.get_object(**written_object)
        with response["Body"] as body:
            assert body.read() == b"deployment diagnostics\n", "Probe bytes differ from the written text."
        print("Probe bytes match.")
        results["S3 probe readback"] = "confirmed"
    except Exception as error:
        print(error)
        results["S3 probe readback"] = str(error)


## 28. Optional ECR repository creation — CreateRepository

- Enable only if creation of this new development repository is permitted.
- Create a unique immutable-tag repository without changing existing repositories.
- Repository creation is independent of permission to push into an existing repository.


In [ ]:
TEST_CREATE_REPOSITORY = False
new_repository_name = probe_name  # Change to the platform's permitted development naming pattern if needed.
print("New repository:", new_repository_name)
results["ECR CreateRepository"] = "not exercised: TEST_CREATE_REPOSITORY is false"
if TEST_CREATE_REPOSITORY:
    try:
        response = ecr.create_repository(repositoryName=new_repository_name, imageTagMutability="IMMUTABLE")
        created_repository = response["repository"]["repositoryName"]
        print(response["repository"]["repositoryUri"])
        results["ECR CreateRepository"] = "confirmed"
    except Exception as error:
        print(error)
        results["ECR CreateRepository"] = str(error)


# Optional: prove an approved base can be pulled, built and pushed

Run this section **on the proposed Docker builder**, not necessarily Studio.
Supply its approved base image and an approved existing ECR repository name,
or use the probe repository just created. Merely discovering a repository
above never selects it for a push. The target is in the current account/region.

This adds one text file to the base image; it does not install model packages.
Success proves this small build/push, not the application's dependencies,
package-mirror access, security scan or managed execution. Docker login uses
the machine's normal credential store. Tokens are passed through stdin and
never printed. The base-image cache and Docker login remain after cleanup.


## 29. Authenticate Docker to the selected ECR registries

- Enable the build test and supply an approved base image and push repository.
- Resolve the target URI and authenticate to it and any private ECR base registry.
- Print Docker login results without printing the authentication token.


In [ ]:
TEST_BUILD_PUSH = False
base_image = ""             # Approved image URI, preferably pinned by digest.
push_repository = ""        # Existing approved repo name; empty uses only a repo created above.
push_repository = push_repository or created_repository
probe_tag = "diagnostics-" + probe_id
results["Docker login"] = "not exercised: TEST_BUILD_PUSH is false"
if TEST_BUILD_PUSH:
    try:
        assert base_image and push_repository, "Supply an approved base image and push repository."
        push_uri = ecr.describe_repositories(repositoryNames=[push_repository])["repositories"][0]["repositoryUri"]
        registries = {push_uri.split("/")[0], base_image.split("/")[0]}
        for registry in sorted(registries):
            match = re.fullmatch(r"(\d{12})\.dkr\.ecr\.([^.]+)\.amazonaws\.com(?:\.cn)?", registry)
            if match:
                registry_client = session.client("ecr", region_name=match[2], config=config)
                auth = registry_client.get_authorization_token()["authorizationData"][0]
                password = base64.b64decode(auth["authorizationToken"]).decode().split(":", 1)[1]
                try:
                    subprocess.run(["docker", "login", "--username", "AWS", "--password-stdin", registry],
                                   input=password, text=True, timeout=30, check=True)
                finally:
                    del auth, password
        print("Push target:", push_uri + ":" + probe_tag)
        results["Docker login"] = "confirmed"
    except Exception as error:
        print(error)
        results["Docker login"] = str(error)


## 30. Pull the approved base image

- Run the literal docker pull command for the supplied image.
- Attempt this even if login failed; existing credentials might already work.
- Print Docker's output or its original command error.


In [ ]:
results["Docker pull base"] = "not exercised: build test off or no base image"
if TEST_BUILD_PUSH and base_image:
    try:
        subprocess.run(["docker", "pull", base_image], check=True, timeout=600)
        results["Docker pull base"] = "confirmed"
    except Exception as error:
        print(error)
        results["Docker pull base"] = str(error)


## 31. Build a tiny image

- Create a temporary context containing a Dockerfile and a harmless text file.
- Build from the supplied base without installing packages or running model code.
- Use a unique tag and remove the temporary source directory when the command finishes.


In [ ]:
results["Docker build"] = "not exercised: build test off or missing image/repository"
if TEST_BUILD_PUSH and base_image and push_uri:
    try:
        with tempfile.TemporaryDirectory(prefix="deployment-check-") as directory:
            context = Path(directory)
            (context / "Dockerfile").write_text("ARG BASE_IMAGE\nFROM ${BASE_IMAGE}\nCOPY probe.txt /deployment-probe.txt\n")
            (context / "probe.txt").write_text(probe_id + "\n")
            image_tag = push_uri + ":" + probe_tag
            subprocess.run(["docker", "build", "--build-arg", "BASE_IMAGE=" + base_image,
                            "--tag", image_tag, str(context)], check=True, timeout=600)
            built_image = image_tag
        results["Docker build"] = "confirmed"
    except Exception as error:
        print(error)
        results["Docker build"] = str(error)


## 32. Push the probe image

- Push only the image successfully built in the previous cell.
- Use the unique diagnostic tag, never latest or an existing model version.
- Print the real push result instead of inferring it from token or list access.


In [ ]:
results["Docker push"] = "not exercised: no probe image built"
if TEST_BUILD_PUSH and built_image:
    try:
        subprocess.run(["docker", "push", built_image], check=True, timeout=600)
        results["Docker push"] = "confirmed"
    except Exception as error:
        print(error)
        results["Docker push"] = str(error)


## 33. Verify the pushed image — DescribeImages

- Look up the exact tag even if the push command reported a failure.
- Print the digest recorded by ECR.
- A lookup denial does not invalidate a successful Docker push; retain both results.


In [ ]:
results["ECR pushed digest"] = "not exercised: no probe image built"
if TEST_BUILD_PUSH and built_image:
    try:
        response = ecr.describe_images(repositoryName=push_repository, imageIds=[{"imageTag": probe_tag}])
        pushed_digest = response["imageDetails"][0]["imageDigest"]
        print(push_uri + "@" + pushed_digest)
        results["ECR pushed digest"] = "confirmed"
    except Exception as error:
        print(error)
        results["ECR pushed digest"] = str(error)


# Optional cleanup

Leave resources for inspection, or enable the following cleanup switches after
finishing. Only resources successfully created in this kernel are eligible.
Keep the printed names if restarting the kernel; this notebook does not search
for and delete earlier probes. Cleanup failures remain visible in the summary.


## 34. Delete this probe pipeline

- Wait for the execution to finish, then enable deletion.
- Delete only this kernel's uniquely named pipeline.
- Preserve existing application pipelines and historical execution records.


In [ ]:
DELETE_PROBE_PIPELINE = False
results["DeletePipeline"] = "not exercised"
if DELETE_PROBE_PIPELINE and pipeline_arn:
    try:
        sm.delete_pipeline(PipelineName=probe_name, ClientRequestToken=probe_id + "-delete")
        print("Deleted:", pipeline_arn)
        pipeline_arn = None
        results["DeletePipeline"] = "confirmed"
    except Exception as error:
        print(error)
        results["DeletePipeline"] = str(error)


## 35. Delete this probe S3 object/version

- Enable deletion only for the object successfully written in this kernel.
- Delete the exact version when bucket versioning supplied one.
- Never delete a discovered input file or a bucket.


In [ ]:
DELETE_PROBE_OBJECT = False
results["S3 DeleteObject"] = "not exercised"
if DELETE_PROBE_OBJECT and written_object:
    try:
        s3.delete_object(**written_object)
        print("Deleted:", written_object)
        written_object = None
        results["S3 DeleteObject"] = "confirmed"
    except Exception as error:
        print(error)
        results["S3 DeleteObject"] = str(error)


## 36. Delete the probe tag and local probe image

- Remove only the unique tag built in this kernel, including a partially successful push.
- Inspect per-image failures returned by ECR.
- Remove the local probe tag; keep the base-image cache and existing Docker authentication.


In [ ]:
DELETE_PROBE_IMAGE = False
results["ECR BatchDeleteImage"] = "not exercised"
if DELETE_PROBE_IMAGE and built_image:
    try:
        response = ecr.batch_delete_image(repositoryName=push_repository, imageIds=[{"imageTag": probe_tag}])
        pprint({"deleted": response.get("imageIds", []), "failures": response.get("failures", [])})
        results["ECR BatchDeleteImage"] = "confirmed" if not response.get("failures") else str(response["failures"])
    except Exception as error:
        print(error)
        results["ECR BatchDeleteImage"] = str(error)
    try:
        subprocess.run(["docker", "image", "rm", built_image], check=True, timeout=30)
    except Exception as error:
        print(error)


## 37. Delete the empty probe repository

- Delete only a repository successfully created in this kernel.
- Use force=False so any remaining image prevents deletion.
- Never delete a discovered or supplied existing repository.


In [ ]:
DELETE_PROBE_REPOSITORY = False
results["ECR DeleteRepository"] = "not exercised"
if DELETE_PROBE_REPOSITORY and created_repository:
    try:
        ecr.delete_repository(repositoryName=created_repository, force=False)
        print("Deleted:", created_repository)
        created_repository = None
        results["ECR DeleteRepository"] = "confirmed"
    except Exception as error:
        print(error)
        results["ECR DeleteRepository"] = str(error)


## 38. One summary to capture

- Print all observed outcomes in one view; full errors remain in the individual cells.
- Save the summary and full result strings locally if desired.
- Keep untested capabilities explicit when discussing access with the platform team.


In [ ]:
SAVE_LOCAL_REPORT = False
lines = [
    "DEPLOYMENT DIAGNOSTICS",
    f"Account: {caller.get('Account', 'unknown')} | Region: {region}",
    f"Caller: {caller.get('Arn', 'unknown')}",
    f"Kernel: {sys.executable}",
    "",
]
for name, outcome in results.items():
    compact = " ".join(outcome.split())
    api_error = re.search(r"An error occurred \(([^)]+)\)", compact)
    if api_error:
        compact = api_error[1] + " (full error in its cell)"
    if len(compact) > 85:
        compact = compact[:82] + "..."
    lines.append(f"{name}: {compact}")
lines.extend([
    "",
    "Not exercised means untested. Discovery reads one page only.",
    "Pipeline probe launches NO JOBS. Runtime job permissions remain untested.",
    "Package downloads, security scans, GitHub build/approval and CDK deploy remain untested.",
    "Docker results apply to this machine/context only.",
])
print("\n".join(lines))
if SAVE_LOCAL_REPORT:
    report_dir = Path.cwd() / ".runs" / probe_name
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "summary.txt").write_text("\n".join(lines) + "\n")
    (report_dir / "details.json").write_text(json.dumps(
        {"account": caller.get("Account"), "caller": caller.get("Arn"),
         "region": region, "results": results}, indent=2))
    print("Local reports:", report_dir.resolve())


To test actual application jobs, **03D_run_managed.ipynb** consumes a published image and
existing resource bindings. **03E_build_and_run_managed.ipynb** demonstrates
the separate option of using an approved existing CodeBuild project.
A Cloud9 or existing GitHub builder can supply the image instead.
